In [2]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize
from scipy.stats import rankdata
import warnings
import os

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

print("lib")


lib


In [3]:

TRAIN_PATH = '../Data/Raw/train.csv' if os.path.exists('../Data/Raw/train.csv') else 'Data/Raw/train.csv'
TEST_PATH = '../Data/Raw/test.csv' if os.path.exists('../Data/Raw/test.csv') else 'Data/Raw/test.csv'

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print(f" Train Shape: {train.shape}")
print(f" Test Shape:  {test.shape}")

print("\n Target Class Distribution ('addicted_label'):")
print(train['addicted_label'].value_counts(normalize=True).map('{:.2%}'.format))

train_nulls = train.isnull().sum().sum()
test_nulls = test.isnull().sum().sum()
print(f"\n Total Missing Values - Train: {train_nulls:,} | Test: {test_nulls:,}")

print("\nData Columns & Types:")
print(train.dtypes)

 Train Shape: (691369, 14)
 Test Shape:  (296302, 13)

 Target Class Distribution ('addicted_label'):
addicted_label
1    70.94%
0    29.06%
Name: proportion, dtype: object

 Total Missing Values - Train: 870,360 | Test: 377,157

Data Columns & Types:
id                           int64
age                        float64
daily_screen_time_hours    float64
social_media_hours         float64
gaming_hours               float64
work_study_hours           float64
sleep_hours                float64
notifications_per_day      float64
app_opens_per_day          float64
weekend_screen_time        float64
gender                      object
stress_level                object
academic_work_impact        object
addicted_label               int64
dtype: object


In [6]:


def build_features(df_train, df_test):
    """
    Extends baseline features with 11 domain ratios, frequency encodings, 
    group aggregations/deviations, log transforms, and clean inf handling.
    """
    df_train = df_train.copy()
    df_test = df_test.copy()
    
    df_train['is_train'] = 1
    df_test['is_train'] = 0
    
    df_full = pd.concat([df_train, df_test], axis=0, ignore_index=True)
    
    eps = 1e-6 
    
    df_full['social_media_ratio'] = df_full['social_media_hours'] / (df_full['daily_screen_time_hours'] + eps)
    df_full['gaming_ratio'] = df_full['gaming_hours'] / (df_full['daily_screen_time_hours'] + eps)
    df_full['work_study_ratio'] = df_full['work_study_hours'] / (df_full['daily_screen_time_hours'] + eps)
    df_full['leisure_hours'] = df_full['daily_screen_time_hours'] - df_full['work_study_hours']
    df_full['leisure_ratio'] = df_full['leisure_hours'] / (df_full['daily_screen_time_hours'] + eps)
    
    df_full['screen_sleep_ratio'] = df_full['daily_screen_time_hours'] / (df_full['sleep_hours'] + eps)
    df_full['weekend_daily_diff'] = df_full['weekend_screen_time'] - df_full['daily_screen_time_hours']
    df_full['weekend_daily_ratio'] = df_full['weekend_screen_time'] / (df_full['daily_screen_time_hours'] + eps)
    df_full['active_non_screen_hours'] = 24.0 - (df_full['daily_screen_time_hours'] + df_full['sleep_hours'])
    
    df_full['notifications_per_screen_hour'] = df_full['notifications_per_day'] / (df_full['daily_screen_time_hours'] + eps)
    df_full['app_opens_per_screen_hour'] = df_full['app_opens_per_day'] / (df_full['daily_screen_time_hours'] + eps)
    df_full['notifications_per_app_open'] = df_full['notifications_per_day'] / (df_full['app_opens_per_day'] + eps)
    
    freq_cols = ['age', 'notifications_per_day', 'app_opens_per_day', 'daily_screen_time_hours']
    for col in freq_cols:
        freq_map = df_full[col].value_counts()
        df_full[f'count_{col}'] = df_full[col].map(freq_map)
        
   
    df_full['age_group'] = pd.qcut(df_full['age'], q=5, labels=False, duplicates='drop')
    
    stress_agg = df_full.groupby(['stress_level', 'academic_work_impact'])['daily_screen_time_hours'].agg(
        mean_screen_time_by_stress='mean',
        std_screen_time_by_stress='std'
    ).reset_index()
    df_full = df_full.merge(stress_agg, on=['stress_level', 'academic_work_impact'], how='left')
    
    df_full['deviation_screen_time_by_stress'] = (
        df_full['daily_screen_time_hours'] - df_full['mean_screen_time_by_stress']
    )
    
    age_agg = df_full.groupby('age_group')['notifications_per_day'].agg(
        mean_notifications_by_age='mean',
        std_notifications_by_age='std'
    ).reset_index()
    df_full = df_full.merge(age_agg, on='age_group', how='left')
    
    df_full['log_notifications'] = np.log1p(np.maximum(0, df_full['notifications_per_day']))
    df_full['log_app_opens'] = np.log1p(np.maximum(0, df_full['app_opens_per_day']))
    
    df_full = df_full.replace([np.inf, -np.inf], np.nan)
    
    train_out = df_full[df_full['is_train'] == 1].drop(columns=['is_train']).reset_index(drop=True)
    test_out = df_full[df_full['is_train'] == 0].drop(columns=['is_train']).reset_index(drop=True)
    
    return train_out, test_out

print(" Building supercharged feature set...")
train_df, test_df = build_features(train, test)

print(f"Feature Engineering Complete!")
print(f"New Train Shape: {train_df.shape}")
print(f"New Test Shape:  {test_df.shape}")

 Building supercharged feature set...
Feature Engineering Complete!
New Train Shape: (691369, 38)
New Test Shape:  (296302, 38)


In [12]:


IGNORE_COLS = ['id', 'addicted_label']
FEATURES = [col for col in train_df.columns if col not in IGNORE_COLS]
CAT_COLS = ['gender', 'stress_level', 'academic_work_impact', 'age_group']

for col in CAT_COLS:
    train_df[col] = train_df[col].astype(str).astype('category')
    test_df[col] = test_df[col].astype(str).astype('category')

X = train_df[FEATURES]
y = train_df['addicted_label'].values
X_test = test_df[FEATURES]

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

print(f"Total Features for Training: {len(FEATURES)}")
print(f"Categorical Features: {CAT_COLS}")
print(f"Training Matrix X: {X.shape} | Target y: {y.shape}")
print(f"Test Matrix X_test: {X_test.shape}")
print(f"StratifiedKFold ready with {N_SPLITS} folds (Random State = {SEED}).")

Total Features for Training: 36
Categorical Features: ['gender', 'stress_level', 'academic_work_impact', 'age_group']
Training Matrix X: (691369, 36) | Target y: (691369,)
Test Matrix X_test: (296302, 36)
StratifiedKFold ready with 5 folds (Random State = 42).


In [9]:

oof_lgb = np.zeros(len(train_df))
test_lgb = np.zeros(len(test_df))

lgb_params = {
    'n_estimators': 2000,
    'learning_rate': 0.035,
    'num_leaves': 45,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': SEED,
    'objective': 'binary',
    'metric': 'auc',
    'n_jobs': -1,
    'verbose': -1
}

print("Starting 5-Fold Cross-Validation for LightGBM...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    X_train, y_train = X.iloc[train_idx], y[train_idx]
    X_val, y_val = X.iloc[val_idx], y[val_idx]
    
    model = lgb.LGBMClassifier(**lgb_params)
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=0)
        ]
    )
    
    val_preds = model.predict_proba(X_val)[:, 1]
    oof_lgb[val_idx] = val_preds
    
    test_lgb += model.predict_proba(X_test)[:, 1] / N_SPLITS
    
    fold_auc = roc_auc_score(y_val, val_preds)
    print(f"  Fold {fold} ROC-AUC: {fold_auc:.6f} | Best Iteration: {model.best_iteration_}")

overall_lgb_auc = roc_auc_score(y, oof_lgb)
print(f"\nLightGBM 5-Fold OOF ROC-AUC Score: {overall_lgb_auc:.6f}")

processed_dir = '../Data/Processed' if os.path.exists('../Data') else 'Data/Processed'
os.makedirs(processed_dir, exist_ok=True)

np.save(os.path.join(processed_dir, 'oof_lgb_attempt5.npy'), oof_lgb)
np.save(os.path.join(processed_dir, 'test_lgb_attempt5.npy'), test_lgb)
print("Saved LightGBM OOF & Test arrays to Data/Processed")

Starting 5-Fold Cross-Validation for LightGBM...
  Fold 1 ROC-AUC: 0.963827 | Best Iteration: 1753
  Fold 2 ROC-AUC: 0.964656 | Best Iteration: 1999
  Fold 3 ROC-AUC: 0.964954 | Best Iteration: 2000
  Fold 4 ROC-AUC: 0.965285 | Best Iteration: 1998
  Fold 5 ROC-AUC: 0.964464 | Best Iteration: 2000

LightGBM 5-Fold OOF ROC-AUC Score: 0.964637
Saved LightGBM OOF & Test arrays to Data/Processed!


In [13]:

oof_xgb = np.zeros(len(train_df))
test_xgb = np.zeros(len(test_df))

xgb_params = {
    'n_estimators': 2000,
    'learning_rate': 0.03,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'enable_categorical': True,
    'tree_method': 'hist',
    'eval_metric': 'auc',
    'random_state': SEED,
    'early_stopping_rounds': 50,
    'n_jobs': -1
}

print(" Starting 5-Fold Cross-Validation for XGBoost...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    X_train, y_train = X.iloc[train_idx], y[train_idx]
    X_val, y_val = X.iloc[val_idx], y[val_idx]
    
    model = xgb.XGBClassifier(**xgb_params)
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    
    val_preds = model.predict_proba(X_val)[:, 1]
    oof_xgb[val_idx] = val_preds
    
    
    test_xgb += model.predict_proba(X_test)[:, 1] / N_SPLITS
    
    fold_auc = roc_auc_score(y_val, val_preds)
    print(f"  Fold {fold} ROC-AUC: {fold_auc:.6f} | Best Iteration: {model.best_iteration}")

overall_xgb_auc = roc_auc_score(y, oof_xgb)
print(f"\nXGBoost 5-Fold OOF ROC-AUC Score: {overall_xgb_auc:.6f}")


np.save(os.path.join(processed_dir, 'oof_xgb_attempt5.npy'), oof_xgb)
np.save(os.path.join(processed_dir, 'test_xgb_attempt5.npy'), test_xgb)
print("Saved XGBoost OOF & Test arrays to Data/Processed")

 Starting 5-Fold Cross-Validation for XGBoost...
  Fold 1 ROC-AUC: 0.964411 | Best Iteration: 1999
  Fold 2 ROC-AUC: 0.965047 | Best Iteration: 1999
  Fold 3 ROC-AUC: 0.965289 | Best Iteration: 1997
  Fold 4 ROC-AUC: 0.965777 | Best Iteration: 1996
  Fold 5 ROC-AUC: 0.964778 | Best Iteration: 1999

XGBoost 5-Fold OOF ROC-AUC Score: 0.965059
Saved XGBoost OOF & Test arrays to Data/Processed


In [14]:

oof_cat = np.zeros(len(train_df))
test_cat = np.zeros(len(test_df))

cat_params = {
    'iterations': 2500,
    'learning_rate': 0.035,
    'depth': 6,
    'eval_metric': 'AUC',
    'random_seed': SEED,
    'early_stopping_rounds': 50,
    'verbose': 250,
    'cat_features': CAT_COLS
}

print("Starting 5-Fold Cross-Validation for CatBoost...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    X_train, y_train = X.iloc[train_idx], y[train_idx]
    X_val, y_val = X.iloc[val_idx], y[val_idx]
    
    model = CatBoostClassifier(**cat_params)
    
    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        use_best_model=True
    )
    
    val_preds = model.predict_proba(X_val)[:, 1]
    oof_cat[val_idx] = val_preds
    
    test_cat += model.predict_proba(X_test)[:, 1] / N_SPLITS
    
    fold_auc = roc_auc_score(y_val, val_preds)
    print(f"  Fold {fold} ROC-AUC: {fold_auc:.6f} | Best Iteration: {model.get_best_iteration()}")

overall_cat_auc = roc_auc_score(y, oof_cat)
print(f"\nCatBoost 5-Fold OOF ROC-AUC Score: {overall_cat_auc:.6f}")

np.save(os.path.join(processed_dir, 'oof_cat_attempt5.npy'), oof_cat)
np.save(os.path.join(processed_dir, 'test_cat_attempt5.npy'), test_cat)
print("Saved CatBoost OOF & Test arrays to Data/Processed")

Starting 5-Fold Cross-Validation for CatBoost...
0:	test: 0.9073928	best: 0.9073928 (0)	total: 684ms	remaining: 28m 29s
250:	test: 0.9464680	best: 0.9464680 (250)	total: 1m 39s	remaining: 14m 49s
500:	test: 0.9530064	best: 0.9530064 (500)	total: 3m 32s	remaining: 14m 9s
750:	test: 0.9564117	best: 0.9564117 (750)	total: 5m 12s	remaining: 12m 7s
1000:	test: 0.9584225	best: 0.9584225 (1000)	total: 6m 45s	remaining: 10m 6s
1250:	test: 0.9597624	best: 0.9597624 (1250)	total: 8m 17s	remaining: 8m 17s
1500:	test: 0.9606726	best: 0.9606726 (1500)	total: 9m 57s	remaining: 6m 37s
1750:	test: 0.9613751	best: 0.9613751 (1750)	total: 11m 38s	remaining: 4m 58s
2000:	test: 0.9619256	best: 0.9619256 (2000)	total: 13m 18s	remaining: 3m 19s
2250:	test: 0.9622872	best: 0.9622872 (2250)	total: 14m 55s	remaining: 1m 39s
2499:	test: 0.9626180	best: 0.9626180 (2499)	total: 16m 38s	remaining: 0us

bestTest = 0.9626180251
bestIteration = 2499

  Fold 1 ROC-AUC: 0.962618 | Best Iteration: 2499
0:	test: 0.908391

In [15]:

oof_df = pd.DataFrame({
    'LGBM': oof_lgb,
    'XGBoost': oof_xgb,
    'CatBoost': oof_cat
})

print("Single Model 5-Fold OOF ROC-AUC Summary:")
print(f"  1. LightGBM: {overall_lgb_auc:.6f}")
print(f"  2. XGBoost:  {overall_xgb_auc:.6f}")
print(f"  3. CatBoost: {overall_cat_auc:.6f}")

print("\n OOF Prediction Pearson Correlation Matrix:")
corr_matrix = oof_df.corr()
print(corr_matrix.round(5))

print("\n Pairwise Diversity Breakdown:")
models = ['LGBM', 'XGBoost', 'CatBoost']
for i in range(len(models)):
    for j in range(i+1, len(models)):
        m1, m2 = models[i], models[j]
        r = corr_matrix.loc[m1, m2]
        print(f"  • {m1:<8} vs {m2:<8} Correlation r = {r:.5f}")

Single Model 5-Fold OOF ROC-AUC Summary:
  1. LightGBM: 0.964637
  2. XGBoost:  0.965059
  3. CatBoost: 0.963231

 OOF Prediction Pearson Correlation Matrix:
             LGBM  XGBoost  CatBoost
LGBM      1.00000  0.99641   0.99339
XGBoost   0.99641  1.00000   0.99440
CatBoost  0.99339  0.99440   1.00000

 Pairwise Diversity Breakdown:
  • LGBM     vs XGBoost  Correlation r = 0.99641
  • LGBM     vs CatBoost Correlation r = 0.99339
  • XGBoost  vs CatBoost Correlation r = 0.99440


In [17]:

def prob_loss_func(weights):
    w1, w2, w3 = weights
    blend = w1 * oof_lgb + w2 * oof_xgb + w3 * oof_cat
   
    return -roc_auc_score(y, blend)

constraints = ({'type': 'eq', 'fun': lambda w: 1.0 - sum(w)})
bounds = [(0.0, 1.0), (0.0, 1.0), (0.0, 1.0)]
init_weights = [1/3, 1/3, 1/3]

res_prob = minimize(
    prob_loss_func,
    init_weights,
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

opt_w1, opt_w2, opt_w3 = res_prob.x
best_prob_auc = -res_prob.fun

print("Probability Blending Optimization Complete")
print(f"  • LightGBM Weight: {opt_w1:.4f}")
print(f"  • XGBoost Weight:  {opt_w2:.4f}")
print(f"  • CatBoost Weight: {opt_w3:.4f}")
print(f"3-Way Weighted Probability Blend OOF ROC-AUC: {best_prob_auc:.6f}")


def to_ranks(arr):
    return rankdata(arr) / len(arr)

oof_lgb_rank = to_ranks(oof_lgb)
oof_xgb_rank = to_ranks(oof_xgb)
oof_cat_rank = to_ranks(oof_cat)

def rank_loss_func(weights):
    w1, w2, w3 = weights
    blend = w1 * oof_lgb_rank + w2 * oof_xgb_rank + w3 * oof_cat_rank
    return -roc_auc_score(y, blend)

res_rank = minimize(
    rank_loss_func,
    init_weights,
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

opt_rw1, opt_rw2, opt_rw3 = res_rank.x
best_rank_auc = -res_rank.fun

print("\nPercentile Rank Averaging Optimization Complete")
print(f"  • LightGBM Rank Weight: {opt_rw1:.4f}")
print(f"  • XGBoost Rank Weight:  {opt_rw2:.4f}")
print(f"  • CatBoost Rank Weight: {opt_rw3:.4f}")
print(f" 3-Way Percentile Rank Blend OOF ROC-AUC: {best_rank_auc:.6f}")

print("\nValidation Performance Comparison:")
best_single = max(overall_lgb_auc, overall_xgb_auc, overall_cat_auc)
print(f"  • Best Single Model OOF:        {best_single:.6f}")
print(f"  • 3-Way Probability Blend OOF:  {best_prob_auc:.6f}")
print(f"  • 3-Way Percentile Rank OOF:     {best_rank_auc:.6f}")

if best_prob_auc >= best_rank_auc:
    print("\nWinning Strategy: Weighted Probability Blending")
    winning_method = 'probability'
    winning_weights = (opt_w1, opt_w2, opt_w3)
    winning_auc = best_prob_auc
else:
    print("\nWinning Strategy: Percentile Rank Averaging")
    winning_method = 'rank'
    winning_weights = (opt_rw1, opt_rw2, opt_rw3)
    winning_auc = best_rank_auc

Probability Blending Optimization Complete
  • LightGBM Weight: 0.3329
  • XGBoost Weight:  0.3342
  • CatBoost Weight: 0.3329
3-Way Weighted Probability Blend OOF ROC-AUC: 0.964840

Percentile Rank Averaging Optimization Complete
  • LightGBM Rank Weight: 0.3333
  • XGBoost Rank Weight:  0.3337
  • CatBoost Rank Weight: 0.3330
 3-Way Percentile Rank Blend OOF ROC-AUC: 0.964829

Validation Performance Comparison:
  • Best Single Model OOF:        0.965059
  • 3-Way Probability Blend OOF:  0.964840
  • 3-Way Percentile Rank OOF:     0.964829

Winning Strategy: Weighted Probability Blending


In [ ]:

w1, w2, w3 = winning_weights

if winning_method == 'probability':
    final_test_preds = w1 * test_lgb + w2 * test_xgb + w3 * test_cat
else:
    test_lgb_rank = to_ranks(test_lgb)
    test_xgb_rank = to_ranks(test_xgb)
    test_cat_rank = to_ranks(test_cat)
    final_test_preds = w1 * test_lgb_rank + w2 * test_xgb_rank + w3 * test_cat_rank

sub = pd.DataFrame({
    'id': test['id'],
    'addicted_label': final_test_preds
})

print("Running Final Submission Integrity Checks...")
print(f"  1. Total Rows:      {len(sub):,} (Expected: 296,302)")
print(f"  2. Header Columns:  {list(sub.columns)}")
print(f"  3. Null Values:     {sub.isnull().sum().sum()}")
print(f"  4. Min Probability: {sub['addicted_label'].min():.6f}")
print(f"  5. Max Probability: {sub['addicted_label'].max():.6f}")
print(f"  6. Mean Prediction: {sub['addicted_label'].mean():.6f}")

assert len(sub) == 296302, "Error: Incorrect row count!"
assert sub.isnull().sum().sum() == 0, "Error: Missing values found!"
assert (sub['addicted_label'] >= 0.0).all() and (sub['addicted_label'] <= 1.0).all(), " Error: Out-of-bounds probabilities"

sub_dir = '../submissions' if os.path.exists('../submissions') else 'submissions'
os.makedirs(sub_dir, exist_ok=True)
sub_path = os.path.join(sub_dir, 'submission_attempt5_master_blend.csv')
sub.to_csv(sub_path, index=False)


os.makedirs(processed_dir, exist_ok=True)
sub.to_csv(os.path.join(processed_dir, 'submission_attempt5_master_blend.csv'), index=False)

print(f"\n Attempt 5 Master Blend Submission exported to:\n   {sub_path}")

Running Final Submission Integrity Checks...
  1. Total Rows:      296,302 (Expected: 296,302)
  2. Header Columns:  ['id', 'addicted_label']
  3. Null Values:     0
  4. Min Probability: 0.000161
  5. Max Probability: 1.000000
  6. Mean Prediction: 0.709611

 Attempt 5 Master Blend Submission exported to:
   submissions\submission_attempt5_master_blend.csv
